# **1. Importar librerías**

In [1]:
# Se importan las librerías que se utilizarán en todo el algoritmo
import numpy as np
import matplotlib.pyplot as plt
import cv2
# En caso de que se desinstale pydicom se debe dejar la línea debajo de este comentario
%pip install pydicom
import pydicom
%pip install gradio
import gradio as gr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.0 MB/s eta 0:00:00


Para este primer bloque, se importaron todas las librerías necesarias para realizar el algoritmo de procesamiento de imágenes médicas, teniendo que utilizar librerías especiales para importar imágenes DICOM y librerías para el diseño de interfaces.

Entre el resto de librerías se encuentran OpenCV, Numpy y Matplotlib para implementar las técnicas de procesamiento digital de imágenes así como su visualización.

# **2. Cargar imagen**

In [2]:
from google.colab import files
archivo = files.upload()
# Almacenamiento de la imagen en una variable
imagen_dicom = pydicom.dcmread("/content/1-001.dcm")
# Se convierte la imagen DICOM a un arreglo de píxeles de Numpy
imagen = imagen_dicom.pixel_array

# Se realiza una normalización de los datos de la imagen original
imagen_modificada = ((imagen.astype(np.float32) - imagen.min()) / (imagen.max() - imagen.min()) * 65535).astype(np.uint16)

Saving 1-001.dcm to 1-001.dcm


En el segundo bloque se importó una imagen médica proporcionada por el instructor. La imagen DICOM se convirtió en un arreglo de píxeles por medio de la librería Numpy y posteriormente se normalizaron los datos que el tipo de dato pasara de "int16" a "uint16", manteniendo los 16 bits de la imagen.

# **3. Análisis y ecualización de histogramas**

In [3]:
# Se realiza un CLAHE para ecualizar la imagen DICOM a 16 bits
clahe = cv2.createCLAHE(clipLimit = 2.0)
imagen_ecualizada = clahe.apply(imagen_modificada)

En el tercer bloque se realiza la ecualización de la imagen modificada al tipo de dato "uint16".

Esta ecualización es por medio de la función "createCLAHE()" el cual tiene mejores resultados que la función "equalizeHist()".

Otro motivo es que la función "equalizeHist()" no acepta imágenes de 16 bits, por lo que para mantener esto, se opta por usar CLAHE.

# **4. Operacions aritméticas**

In [4]:
# Se le aplica un filtro paso bajo a la imagen ecualizada
imagen_filtrada = cv2.GaussianBlur(imagen_ecualizada, (9,9), 0)

# Se hace la resta entre la imagen original y la imagen suavizada
imagen_resta = cv2.subtract(imagen_ecualizada, imagen_filtrada)

Para el cuarto bloque se realizó una resta con la función "substract()" entre la imagen ecualizada con una imagen filtrada por medio de un filtro paso bajo (Gaussiano) con la función "GaussianBlur()". El próposito de esto es para resaltar los bordes de la imagen médica.

# **5. Magnitud del espectro de Fourier**

In [5]:
# Transformada discreta de Fourier aplicado en la imagen
Fourier = np.fft.fft2(imagen_ecualizada)
Fourier = np.fft.fftshift(Fourier)

# Obteniendo la magnitud del espectro de Fourier, sacando su logaritmo debido a los valores elevados
Magnitud = np.log(np.abs(Fourier))

El quinto bloque consistió en obtener la magnitud del espectro de Fourier por medio de la transformada discreta de Fourier con la función "fft.fft2()" el cual convierte una imagen al dominio de la frecuencia.

Posteriormente se utilizó la función "fft.fftshift()" para colocar la frecuenica 0 (o en general las frecuencias bajas) en el centro del espectro.

Por último, se obtuvo la magnitud por medio de la función "abs()" y se calculó el logaritmo natural de dicho resultado para obtener valores más pequeños y que puedan ser visualizados.

# **6. Diseño de interfaz**

In [6]:
# Preparacion de la imagen para la pagina Web
# La convierte en formato de 8 bits para que cualquier navegador lo pueda leer
def preparar_imagen_web(img_array):
    """Convierte el arreglo procesado a un formato amigable para visualización web"""
    return cv2.normalize(img_array, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

def generar_histograma(img_array, titulo):
    """Crea una figura de matplotlib para incrustarla en la interfaz web"""
    fig = plt.figure(figsize=(6, 4))
    plt.hist(img_array.ravel(), bins=255, color='gray')
    plt.title(titulo)
    plt.xlabel("Intensidad de píxeles")
    plt.ylabel("Número de píxeles")
    return fig

# Las funciones Wrappers
# Gradio no funciona con "plt.show" por lo que utilizas estas funciones como intermediarias, reciben los valores que el usario escoga en la pagina web
# En vez de imprimir la imagen procesan los datos aqui en el colab y luego la imagen final la mandan de regreso a la pagina web
def gradio_logicas(u1, u2, op1, op2):
    primera_mascara = cv2.threshold(imagen_ecualizada, u1, 65535, cv2.THRESH_BINARY)[1]
    segunda_mascara = cv2.threshold(imagen_ecualizada, u2, 65535, cv2.THRESH_BINARY)[1]

    if op1 == "AND": resultado = cv2.bitwise_and(primera_mascara, segunda_mascara)
    elif op1 == "OR": resultado = cv2.bitwise_or(primera_mascara, segunda_mascara)
    elif op1 == "XOR": resultado = cv2.bitwise_xor(primera_mascara, segunda_mascara)
    elif op1 == "NOT":
        if op2 == "Primera máscara": resultado = cv2.bitwise_not(primera_mascara)
        else: resultado = cv2.bitwise_not(segunda_mascara)

    return preparar_imagen_web(resultado)

def gradio_filtros(opcion):
    if opcion == "Filtro Gaussiano": resultado = cv2.GaussianBlur(imagen_ecualizada, (5,5), 0)
    elif opcion == "Filtro de Media": resultado = cv2.blur(imagen_ecualizada, (5,5))
    elif opcion == "Filtro Laplaciano": resultado = cv2.Laplacian(imagen_ecualizada, cv2.CV_32F, ksize = 5)
    elif opcion == "Filtro Sobel":
        sobel_x = cv2.Sobel(imagen_ecualizada, cv2.CV_32F, 1, 0, ksize = 5)
        sobel_y = cv2.Sobel(imagen_ecualizada, cv2.CV_32F, 0, 1, ksize = 5)
        resultado = cv2.magnitude(sobel_x, sobel_y)

    return preparar_imagen_web(resultado)

def gradio_segmentacion(valor):
    umbral = cv2.threshold(imagen_ecualizada, valor, 65535, cv2.THRESH_BINARY)[1]
    return preparar_imagen_web(umbral)

def gradio_fourier(radio):
    fila, columna = imagen_ecualizada.shape
    Mascara = np.zeros((fila, columna), dtype = np.float32)
    y, x = np.ogrid[:fila, :columna]
    distancia = np.sqrt((x - columna//2) ** 2 + (y - fila//2) ** 2)
    Mascara[distancia <= radio] = 1

    Fourier_filtrado = Fourier * Mascara
    Imagen_filtrada = np.abs(np.fft.ifft2(np.fft.ifftshift(Fourier_filtrado)))

    return preparar_imagen_web(Imagen_filtrada)

# Construccion de todo el Dashboard
# Utiliza un sistema llamado Blocks para diseñar la pagina web
with gr.Blocks(title="Análisis de Imágenes Médicas", theme=gr.themes.Soft()) as dashboard:
    gr.Markdown("# Dashboard")
    gr.Markdown("Ajusta los parámetros en tiempo real para resaltar estructuras patológicas o anatómicas.")

    # Sistema de navegacion de pestañas y acomodo de los objetos en la pagina web
    with gr.Tabs():

        with gr.TabItem("Histogramas"):
            with gr.Row():
                gr.Plot(value=generar_histograma(imagen, "Histograma Original"), label="Original")
                gr.Plot(value=generar_histograma(imagen_ecualizada, "Histograma Ecualizado"), label="Ecualizado")

        with gr.TabItem("Imagen Medica"):
            with gr.Row():
                gr.Image(value=preparar_imagen_web(imagen), label="Imagen Médica Original", image_mode="L")
                gr.Image(value=preparar_imagen_web(imagen_ecualizada), label="Imagen Médica Ecualizada", image_mode="L")

        with gr.TabItem("Operaciones Aritmeticas"):
            gr.Markdown("### Imagen Original - Imagen Suavizada")
            gr.Image(value=preparar_imagen_web(imagen_resta), label="Resultado de la Resta", image_mode="L")

        # Operaciones Lógicas
        with gr.TabItem("Operaciones Lógicas"):
            with gr.Row():
                with gr.Column():
                    u1_in = gr.Slider(0, 65535, value=30000, step=1, label="Umbral 1")
                    u2_in = gr.Slider(0, 65535, value=30000, step=1, label="Umbral 2")
                    op1_in = gr.Dropdown(["AND", "OR", "XOR", "NOT"], value="AND", label="Operador Lógico")
                    op2_in = gr.Dropdown(["Primera máscara", "Segunda máscara"], value="Primera máscara", label="Objetivo del NOT")

                # Se carga la imagen inicial evaluando la función con los valores por defecto
                img_out_logicas = gr.Image(value=gradio_logicas(30000, 30000, "AND", "Primera máscara"),
                                           label="Resultado Lógico", image_mode="L")

            # Conectar entradas con salida para cuando el usuario interactúe
            inputs_logicas = [u1_in, u2_in, op1_in, op2_in]
            for inp in inputs_logicas:
                inp.change(fn=gradio_logicas, inputs=inputs_logicas, outputs=img_out_logicas)

        # Filtros Espaciales
        with gr.TabItem("Filtros Espaciales"):
            with gr.Row():
                with gr.Column():
                    filtro_in = gr.Dropdown(["Filtro Gaussiano", "Filtro de Media", "Filtro Laplaciano", "Filtro Sobel"],
                                            value="Filtro Gaussiano", label="Seleccionar Filtro")

                # Imagen inicial para el filtro
                img_out_filtros = gr.Image(value=gradio_filtros("Filtro Gaussiano"),
                                           label="Resultado del Filtro", image_mode="L")

            filtro_in.change(fn=gradio_filtros, inputs=filtro_in, outputs=img_out_filtros)

        # Segmentación
        with gr.TabItem("Segmentación"):
            with gr.Row():
                with gr.Column():
                    seg_in = gr.Slider(0, 65535, value=30000, step=1, label="Umbral de Segmentación")

                # Imagen inicial para la segmentación
                img_out_seg = gr.Image(value=gradio_segmentacion(30000),
                                       label="Imagen Segmentada", image_mode="L")

            seg_in.change(fn=gradio_segmentacion, inputs=seg_in, outputs=img_out_seg)

        # Filtro Fourier
        with gr.TabItem("Filtro Fourier"):
            with gr.Row():
                with gr.Column():
                    radio_in = gr.Slider(0, 100, value=30, step=1, label="Radio del Filtro")

                # Imagen inicial para Fourier
                img_out_fourier = gr.Image(value=gradio_fourier(30),
                                           label="Espectro Filtrado", image_mode="L")

            # Esto hace que la interfaz responda cuando el usario realiza algo en la pagina web
            radio_in.change(fn=gradio_fourier, inputs=radio_in, outputs=img_out_fourier)

        with gr.TabItem("Magnitud Fourier"):
            gr.Markdown("### Espectro de Magnitud (Logaritmo)")
            gr.Image(value=preparar_imagen_web(Magnitud), label="Magnitud de Fourier Base", image_mode="L")


# Ejecuta la aplicación web (genera el link para abrir en otra pestaña)
dashboard.launch(debug=True)

/tmp/ipykernel_1901/1236792890.py:61: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Análisis de Imágenes Médicas", theme=gr.themes.Soft()) as dashboard:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0b34dc6f8455a9b3ca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0b34dc6f8455a9b3ca.gradio.live


Este último bloque representa la construccion de la interfaz propuesta para la manipulación de la imagen médica, mostrando el resultado de los puntos anteriores mencionados así como la aplicación de máscaras binarias usando operaciones lógicas, la aplicación de filtros para la eliminación de ruido y detección de bordes, la segmentación de la imagen por medio de un umbral adaptativo y la aplicación de un filtro paso bajo en el dominio de la frecuencia.

Todos estos puntos mencionados se realizaron con funciones correspondientes a las librerías de OpenCV, Numpy y Matplotlib.

Asimismo, la librería Gradio se utilizó completamente en este bloque para el diseño de la interfaz.